# FAIS PRATAMA
# H061251055
# AI GEOFISIKA (B)

------------------------------------------------------------------------

# BAGIAN I --- RECIPE DARI BUKU

## 1. Recipe Audit Station CSV

### Problem

Baca satu file CSV stasiun dan identifikasi masalah pada skema kolom,
rentang waktu, nilai hilang, serta timestamp duplikat. Hasilnya berupa
ringkasan QA/QC, bukan analisis iklim atau prediksi hujan. File sumber
tidak boleh diubah.

### Minimal Theory

Audit data dilakukan sebelum agregasi atau analisis agar kesalahan
struktur, waktu, dan nilai dapat diketahui lebih awal. Timestamp ganda
dapat memengaruhi agregasi, sedangkan nilai hilang atau tipe kolom yang
tidak sesuai dapat mengubah hasil analisis.

### Input dan Constraints

-   `CSV_PATH`: path file CSV.
-   `TIME_COLUMN`: nama kolom waktu.
-   `UNITS`: kontrak satuan untuk kolom yang relevan.
-   Pertahankan file sumber; gunakan pembacaan *read-only*.
-   Jangan menghapus atau memperbaiki baris sumber secara otomatis.
-   Jangan menyimpulkan hasil ilmiah yang tidak diuji oleh audit.

### Prompt dan Procedure

> Buat draft recipe untuk audit QA/QC station CSV. Periksa skema kolom,
> rentang waktu, nilai hilang, timestamp duplikat, dan tipe data tanpa
> mengubah input mentah. Nyatakan input, teori minimum, asumsi,
> prosedur, output yang diharapkan, rencana implementasi Python,
> pemeriksaan kegagalan, serta status eksekusi. Pisahkan spesifikasi
> recipe dari klaim bahwa program telah dijalankan.

Langkah kerja: 1. Baca CSV secara *read-only*. 2. Periksa apakah kolom
wajib tersedia. 3. Validasi tipe data dan rentang timestamp. 4. Hitung
nilai hilang dan timestamp duplikat. 5. Buat laporan QA/QC dan catatan
pemeriksaan. 6. Catat error atau kolom yang tidak sesuai tanpa mengubah
sumber.

### Code / Implementation Plan

Implementasi dapat menggunakan Python dengan `pandas` untuk membaca CSV
dan menghitung ringkasan. Kode disimpan pada path relatif, misalnya
`code/recipes/io/r-io-001.py`. Path ini merupakan rancangan, bukan bukti
bahwa file sudah dibuat.

### Output

-   `results/io/station-summary.md`: laporan Markdown.
-   `results/run-records/r-io-001.json`: run record jika program
    dijalankan.

Laporan mencakup jumlah baris dan kolom, kolom yang ditemukan, rentang
waktu, jumlah nilai hilang, jumlah timestamp duplikat, dan status setiap
pemeriksaan.

### Optional Check / Evidence

Periksa bahwa file input tidak berubah, laporan tidak kosong, jumlah
baris sesuai dengan sumber, dan status eksekusi dicatat. Gunakan
`Not yet executed` jika program belum dijalankan. Label `Executed` hanya
digunakan jika bukti eksekusi tersedia.

------------------------------------------------------------------------

## 2. Recipe Normalisasi GeoJSON

### Problem

Susun recipe untuk mengubah file GeoJSON menjadi GeoJSON yang
dinormalisasi, inventaris CSV, dan ringkasan JSON. Recipe harus
memeriksa geometri, properti, geometri kosong, serta urutan koordinat
tanpa menebak informasi yang tidak tersedia.

### Minimal Theory

GeoJSON menggunakan struktur JSON untuk menyimpan fitur geografis.
Pemeriksaan perlu mencakup struktur `FeatureCollection`, tipe geometri,
koordinat, properti, dan sistem referensi koordinat. Untuk koordinat
geografis GeoJSON, urutan koordinat umumnya berupa bujur lalu lintang.

### Input dan Constraints

-   `GEOJSON_PATH`: path GeoJSON sumber.
-   Path output untuk GeoJSON, CSV inventaris, dan JSON ringkasan.
-   Jangan menghilangkan fitur atau properti tanpa aturan yang jelas.
-   Jangan menebak CRS atau mengubah urutan koordinat tanpa dasar.
-   Keberhasilan membaca file tidak otomatis membuktikan topologi
    geometri valid.

### Prompt dan Procedure

> Susun recipe untuk menormalisasi `[GEOJSON_PATH]` menjadi GeoJSON,
> inventaris CSV, dan ringkasan JSON. Periksa struktur, tipe geometri,
> properti, geometri kosong atau tidak valid, urutan koordinat, dan
> informasi CRS. Nyatakan ketidakpastian secara terbuka dan jangan
> mengklaim output telah dihasilkan sebelum eksekusi dibuktikan.

Langkah kerja: 1. Baca dan validasi sintaks JSON. 2. Periksa struktur
objek dan daftar fitur. 3. Periksa geometri, koordinat, serta properti.
4. Identifikasi CRS atau urutan koordinat yang belum jelas. 5.
Normalisasi hanya berdasarkan aturan yang terdefinisi. 6. Ekspor
inventaris fitur dan ringkasan hasil pemeriksaan. 7. Catat fitur yang
gagal diproses.

### Code / Implementation Plan

Implementasi dapat menggunakan Python dengan pustaka yang mendukung
GeoJSON, misalnya `json` untuk pemeriksaan struktur dan pustaka
geospasial yang tersedia di lingkungan kerja. Dependency harus dicatat
dan tidak boleh diasumsikan sudah terpasang.

### Output

-   GeoJSON hasil normalisasi.
-   CSV inventaris fitur.
-   JSON ringkasan QA/QC.
-   Run record serta peringatan jika tersedia.

### Optional Check / Evidence

Bandingkan jumlah fitur sumber dan hasil, periksa properti penting,
validasi struktur output, dan catat fitur yang tidak berhasil diproses.
Jika program belum dijalankan, sebutkan output sebagai hasil yang
direncanakan.

------------------------------------------------------------------------

## 3. Recipe Model Difusi Panas Satu Dimensi

### Problem

Buat recipe untuk mensimulasikan perubahan suhu terhadap posisi dan
waktu berdasarkan profil awal, difusivitas termal, kondisi batas, jarak
grid, dan langkah waktu.

### Minimal Theory

Persamaan difusi panas satu dimensi adalah:

\[\frac{\partial T}{\partial t}=\alpha\frac{\partial^2T}{\partial x^2}\]

dengan \(T\) adalah suhu, \(t\) waktu, \(x\) posisi, dan \(\alpha\)
difusivitas termal.

Untuk skema beda hingga eksplisit standar, parameter kestabilan dapat
ditulis:

\[r=\frac{\alpha\Delta t}{(\Delta x)^2}\]

Pada kasus satu dimensi dengan skema eksplisit standar, syarat
kestabilan yang umum digunakan adalah \(r \le 1/2\). Syarat ini
perlu disesuaikan dengan skema numerik yang benar-benar digunakan.

### Input dan Constraints

-   `INITIAL_PROFILE`: profil suhu awal.
-   `ALPHA`: difusivitas termal dan satuannya.
-   `BOUNDARIES`: kondisi batas.
-   `DZ`: jarak grid.
-   `DT`: langkah waktu.
-   Pastikan konsistensi satuan.
-   Jangan mengklaim kalibrasi atau konvergensi tanpa pengujian.

### Prompt dan Procedure

> Buat draft recipe model difusi panas satu dimensi menggunakan profil
> awal `[INITIAL_PROFILE]`, difusivitas `[ALPHA]`, kondisi batas
> `[BOUNDARIES]`, jarak grid `[DZ]`, dan langkah waktu `[DT]`. Sertakan
> persamaan pengatur, konvensi satuan dan tanda, pemeriksaan kestabilan,
> pembanding analitik atau kasus dasar, profil output, grafik, run
> record, serta peringatan jika parameter atau dependency belum
> tersedia. Jangan menyatakan eksekusi tanpa bukti.

Langkah kerja: 1. Validasi parameter dan satuan. 2. Bentuk grid ruang
dan waktu. 3. Terapkan kondisi awal serta kondisi batas. 4. Periksa
kestabilan skema. 5. Jalankan iterasi jika implementasi tersedia. 6.
Bandingkan hasil dengan kasus dasar atau solusi analitik jika
memungkinkan. 7. Simpan profil suhu, grafik, dan catatan eksekusi.

### Code / Implementation Plan

Implementasi dapat menggunakan Python, `numpy` untuk komputasi numerik,
dan `matplotlib` for visualisasi. Pilihan metode numerik, parameter,
serta dependency harus dicatat.

### Output

-   Profil suhu terhadap posisi dan waktu.
-   Grafik hasil simulasi.
-   Ringkasan parameter.
-   Run record berisi status dan peringatan.

### Optional Check / Evidence

Periksa kestabilan numerik, kondisi batas, konsistensi satuan, dan
kesesuaian dengan kasus dasar. Jangan mengklaim konvergensi, kalibrasi,
atau keberhasilan eksekusi tanpa bukti yang sesuai.

------------------------------------------------------------------------

# BAGIAN II --- RECIPE BARU BUATAN SAYA

## 4. Recipe QC dan Analisis Kurva HVSR Mikroseismik 3-Komponen (Geofisika)

### Problem

Membuat *workflow* terukur dan *reproducible* untuk melakukan kontrol kualitas (QA/QC) sinyal mikroseismik 3-komponen (Sumbu Vertikal Z, Utara-Selatan N, Timur-Barat E), pemotongan jendela waktu (*time windowing*), penghitungan Spektrum Amplitudo HVSR (*Horizontal-to-Vertical Spectral Ratio*), serta penentuan frekuensi dominan tanah (\(f_0\)) dan faktor amplifikasi (\(A_0\)) sesuai standar SESAME (2004).

Masalah yang diperiksa mencakup *clipping* sinyal, nilai *offset* DC/drift, rasio sinyal terhadap bising (*SNR*), gangguan gelombang lokal (*transient noise* akibat lalu lintas/aktivitas manusia), ketidaksesuaian panjang sampel antar komponen, serta kepatuhan terhadap kriteria keandalan (*reliability criteria*) dan kriteria kejelasan puncak (*prominence criteria*) SESAME.

### Minimal Theory

Metode HVSR mengestimasi respon tapak lokal (*local site response*) dengan membandingkan spektrum amplitudo komponen horizontal terhadap komponen vertikal dari rekaman mikroseismik/bising lingkungan:

\[\text{HVSR}(f) = \frac{H(f)}{V(f)} = \frac{\sqrt{S_{NS}^2(f) + S_{EW}^2(f)}}{\sqrt{2} \cdot S_{Z}(f)}\]

dengan \(S_{NS}(f)\), \(S_{EW}(f)\), dan \(S_Z(f)\) adalah Spektrum Amplitudo Fourier dari komponen Utara-Selatan, Timur-Barat, dan Vertikal setelah dihaluskan (*smoothed*) menggunakan *Konno-Ohmachi smoothing filter*.

Berdasarkan pedoman SESAME (2004), puncak HVSR pada frekuensi \(f_0\) dianggap andal jika memenuhi kondisi:
1. Puncak berada pada frekuensi \(f_0 > 10 / N_w \cdot l_w\) (dengan \(N_w\) jumlah jendela dan \(l_w\) panjang jendela).
2. Jumlah siklus \(N_c = N_w \cdot l_w \cdot f_0 > 200\).
3. Standar deviasi \(\sigma_A(f)\) pada rentang frekuensi puncak berada di bawah ambang batas kritis tertentu.

Frekuensi puncak \(f_0\) berhubungan langsung dengan ketebalan lapisan sedimen (\(H\)) dan kecepatan gelombang geser (\(V_s\)):

\[f_0 = \frac{V_s}{4H}\]

### Input dan Constraints

- `WAVEFORM_PATH`: lokasi file rekaman mikroseismik (format Miniseed, SAC, atau CSV).
- `SAMPLING_RATE`: frekuensi sampling rekaman dalam Hz (misal: 100 Hz).
- `WINDOW_LENGTH`: durasi jendela waktu dalam detik (misal: 20–40 detik).
- `SMOOTHING_BANDWIDTH`: konstanta penghalusan Konno-Ohmachi (misal: \(b = 40\)).
- `FREQ_RANGE`: rentang frekuensi analisis (misal: 0.2–20 Hz).

Batasan:
1. File mentah berstatus *read-only* tanpa mengubah isi sinyal asli.
2. Jendela waktu yang terinfeksi gelombang *transient* (misal: orang berjalan dekat sensor) harus dieliminasi menggunakan algoritma *anti-trigger* (misal: ST/LT ratio).
3. Sinyal yang mengalami *clipping* atau *saturation* harus ditandai dan dikeluarkan dari komputasi.
4. Jangan mengklaim estimasi kedalaman (*bedrock depth*) secara definitif tanpa inversi nilai \(V_s\) atau data sumur bor (*borehole*).

### Prompt dan Procedure

**Prompt yang dapat digunakan kembali:**

> Anda bertindak sebagai asisten pengolahan data geofisika mikroseismik. Baca berkas rekaman 3-komponen pada `[WAVEFORM_PATH]` secara read-only. Periksa kelengkapan komponen (Z, N, E), frekuensi sampling, keberadaan offset DC, serta ketersediaan data.
>
> Buat laporan QA/QC sinyal mentah. Lakukan pra-pemrosesan (detrend, taper, bandpass filter), bagi sinyal menjadi jendela-jendela waktu pendek, eliminasikan jendela yang mengalami gangguan transient noise, lalu hitung spektrum HVSR dengan penghalusan Konno-Ohmachi.
>
> Tampilkan grafik kurva rata-rata HVSR beserta pita standar deviasi (\(\pm 1\sigma\)), identifikasi nilai \(f_0\) dan \(A_0\), serta lakukan verifikasi kriteria keandalan SESAME. Catat hasil ke dalam laporan Markdown dan run record JSON. Tulis status Not yet executed jika kode belum dieksekusi.

**Procedure:**
1. Baca file rekaman mikroseismik secara *read-only*.
2. Lakukan audit sinyal mentah: periksa *data completeness*, selisih sampel, *clipping*, dan *drift*.
3. Aplikasikan pra-pemrosesan dasar: *mean removal* (detrending) dan *cosine tapering* pada kedua ujung sinyal.
4. Pembagian jendela waktu (*windowing*) serta eliminasi *transient noise* berdasarkan analisis STA/LTA.
5. Hitung Fast Fourier Transform (FFT) pada tiap jendela untuk komponen Z, N, dan E.
6. Aplikasikan penghalusan spektrum (*Konno-Ohmachi smoothing filter*).
7. Hitung rasio H/V per jendela, lalu hitung nilai rata-rata (*log-average*) dan standar deviasinya.
8. Uji kriteria keandalan dan kejelasan puncak sesuai standar SESAME (2004).
9. Plot kurva HVSR (Frekuensi vs Rasio H/V) beserta rentang deviasi.
10. Susun laporan QC, ringkasan parameter mikroseismik (\(f_0\), \(A_0\)), dan simpan *run record*.

### Code / Implementation Plan

Rencana implementasi menggunakan Python:
- `obspy`: pembacaan format seismogram (MiniSEED/SAC) dan pra-pemrosesan sinyal.
- `numpy` & `scipy`: transformasi Fourier (FFT), manipulasi matriks, dan pemfilteran sinyal.
- `matplotlib`: visualisasi kurva HVSR, spektrogram, dan perbandingan sinyal waktu.
- `json` & `pathlib`: penyimpan meta-data, parameter kontrol, dan catatan eksekusi.

Rancangan struktur direktori proyek:

```text
hvsr-microseismic-qc/
├── data/
│   └── raw-waveform.mseed
├── code/
│   └── hvsr_processing_qc.py
└── results/
    ├── hvsr-qc-report.md
    ├── hvsr-curve-plot.png
    ├── hvsr-summary-parameters.csv
    └── run-record.json

#Struktur di atas merupakan acuan perancangan. Keberadaan berkas fisik harus diverifikasi secara terpisah setelah eksekusi dilakukan.

### Output

1. `hvsr-qc-report.md`: laporan QA/QC sinyal mentah, statistik jendela waktu terpakai vs terbuang, serta status kriteria kelayakan SESAME.
2. `hvsr-curve-plot.png`: grafik log-log / semi-log kurva H/V (Rasio Amplitudo vs Frekuensi) beserta garis batas standar deviasi \(\pm 1\sigma\) dan penanda frekuensi puncak (\(f_0\)).
3. `hvsr-summary-parameters.csv`: tabel estimasi nilai frekuensi dasar (\(f_0\)), amplitudo puncak (\(A_0\)), dan rentang deviasi.
4. `run-record.json`: catatan eksekusi berisi konfigurasi parameter, versi pustaka, waktu eksekusi, serta log peringatan sistem.

### Optional Check / Evidence

Checklist pemeriksaan:
- [ ] File rekaman sumber bersifat *read-only* tanpa modifikasi fisik.
- [ ] Tiga komponen (Z, N, E) lengkap dan memiliki frekuensi sampling yang seragam.
- [ ] Pemotongan jendela waktu mengeliminasi sinyal *transient noise* beramplitudo tinggi.
- [ ] Penghalusan spektrum Konno-Ohmachi diterapkan sesuai parameter yang ditentukan.
- [ ] Puncak frekuensi (\(f_0\)) diuji terhadap kriteria SESAME (2004).
- [ ] Ambiguitas hubungan \(f_0\) terhadap kedalaman sedimen dinyatakan terbuka dalam laporan.
- [ ] Status eksekusi ditandai secara akurat (*Draft* / *Not yet executed* / *Executed*).

Status eksekusi yang diperbolehkan:
- `Draft`: spesifikasi recipe sedang dirancang.
- `Not yet executed`: skrip/program pemroses belum dijalankan.
- `Partial`: pra-pemrosesan sinyal selesai, namun komputasi HVSR belum berjalan.
- `Executed`: skrip telah dieksekusi penuh dan seluruh berkas output tersedia.

### Keterbatasan dan Risiko

Metode HVSR efektif untuk mengestimasi frekuensi resonansi tanah, namun memiliki keterbatasan fisis dalam menentukan profil kecepatan gelombang geser (\(V_s\)) secara tunggal (*non-uniqueness problem*). Nilai amplitudo puncak (\(A_0\)) tidak secara langsung sama dengan faktor amplifikasi situs yang sebenarnya saat gempa kuat terjadi (*non-linear site response*). Estimasi ketebalan sedimen dari \(f_0\) tanpa validasi kecepatan \(V_s\) melalui metode MASW, inversi kurva dispersi, atau data pemboran teknik berisiko menghasilkan kesalahan interpretasi geologi teknik yang signifikan.

### Ringkasan Recipe Baru

Recipe ini menyusun kerangka kerja standar dalam analisis QA/QC dan komputasi spektrum HVSR dari data rekaman mikroseismik 3-komponen. Dimulai dari audit kelayakan sinyal mentah, penapisan *transient noise*, kalkulasi spektrum Fourier terhaluskan, hingga validasi keandalan puncak sesuai standar internasional SESAME (2004). Recipe ini memastikan proses olah data mikroseismik bersifat transparan, dapat direproduksi (*reproducible*), dan meminimalkan bias interpretasi subjektif.